In [1]:
import os
os.environ["R_HOME"] = "/homes/kbalzer/miniconda3/envs/qiime2-amplicon-2026.1/lib/R"

qiime_env = "/homes/kbalzer/miniconda3/envs/qiime2-amplicon-2026.1"
os.environ["PATH"] = f"{qiime_env}/bin:/usr/local/bin:/usr/bin:/bin"
os.environ["CONDA_PREFIX"] = qiime_env


In [2]:
from ancombc2_heatmaps import (
    PlotWorkflow,
    HeatmapConfig,
    MetadataConfig,
    ComparisonConfig,
    PathConfig,
    SubsetSpec,
    TrajectoryConfig,
    TrajectoryMetadataConfig,
    TrajectoryPathConfig,
    TrajectoryPlotConfig,
)

print("Imports ok")

Imports ok


In [3]:
TIMEPOINTS = [
    "baseline1", "baseline2", "baseline3",
    "day1", "day3", "day7", "day14"
]

TIMEPOINT_MAP = {
    "baseline_1": "baseline1",
    "baseline_2": "baseline2",
    "baseline_3": "baseline3",
    "day_1_post": "day1",
    "day_3_post": "day3",
    "day_7_post": "day7",
    "day_14_post": "day14",
    "baseline1": "baseline1",
    "baseline2": "baseline2",
    "baseline3": "baseline3",
    "day1": "day1",
    "day3": "day3",
    "day7": "day7",
    "day14": "day14",
}

TIMEPOINT_NUMERIC_MAP = {
    "baseline_1": -7,
    "baseline_2": -4,
    "baseline_3": -1,
    "day_1_post": 1,
    "day_3_post": 3,
    "day_7_post": 7,
    "day_14_post": 14,
    "baseline1": -7,
    "baseline2": -4,
    "baseline3": -1,
    "day1": 1,
    "day3": 3,
    "day7": 7,
    "day14": 14,
}

TP_LABEL_MAP = {
    0: "baseline",
    -7: "baseline1",
    -4: "baseline2",
    -1: "baseline3",
    1: "day1",
    3: "day3",
    7: "day7",
    14: "day14",
}

In [4]:
META_FP = "/vol/jlab/MicrobiomeAnalyses/Projects/JansenVanVuuren_Microbian2/Karl/metadata_microbian2_26.02.2026.txt"

TABLE_BASE = "/vol/jlab/MicrobiomeAnalyses/Projects/JansenVanVuuren_Microbian2/Karl/heatmaps_genus_by_timepoint/by_sex"

ANCOM_BASE = "/vol/jlab/MicrobiomeAnalyses/Projects/JansenVanVuuren_Microbian2/Karl/heatmaps_genus_by_timepoint/by_sex/ancombc2_sex"

OUTDIR = "/tmp/ancombc2_notebook_test"

In [5]:
heatmap_config = HeatmapConfig(
    metadata=MetadataConfig(
        sample_col="sample_name",
        timepoint_col="time_point",
        comparison_col="sex",
        timepoints=TIMEPOINTS,
        timepoint_map=TIMEPOINT_MAP,
        allowed_values={
            "description_of_treatment": ["sham", "irradiated"],
            "sex": ["male", "female"],
        },
    ),
    comparison=ComparisonConfig(
        variable_name="sex",
        positive_class="male",
        negative_class="female",
    ),
    paths=PathConfig(
        base_table_dir=TABLE_BASE,
        base_ancom_dir=ANCOM_BASE,
        metadata_path=META_FP,
        output_dir=OUTDIR,
        table_template="{timepoint}/table_{timepoint}_{subset_label}.qza",
        ancom_template="{timepoint}/table_{timepoint}_{subset_label}_sex_ANCOMB_exported",
    ),
    cell_text_mode="relative_abundance",   # "relative_abundance", "lfc", "none"
    split_after_timepoint="baseline3",
    min_sig_cells_per_taxon=1,
)

print("Heatmap config ok")

Heatmap config ok


In [6]:

traj_config = TrajectoryConfig(
    metadata=TrajectoryMetadataConfig(
        sample_col="sample_name",
        timepoint_col="time_point",
        mouse_col="host_subject_id",
        comparison_col="sex",
        genotype_col="mice_model",
        treatment_col="description_of_treatment",
        timepoint_order=TIMEPOINTS,
        timepoint_numeric_map=TIMEPOINT_NUMERIC_MAP,
        timepoint_label_map=TP_LABEL_MAP,
        allowed_values={
            "description_of_treatment": ["sham", "irradiated"],
            "sex": ["female", "male"],
        },
    ),
    paths=TrajectoryPathConfig(
        metadata_path=META_FP,
        table_base=TABLE_BASE,
        ancom_base=ANCOM_BASE,
    ),
    plot=TrajectoryPlotConfig(
        estimator="mean",                 # "mean" or "median"
        error_style="iqr",                # "iqr" or "ci"
        show_individual_lines=True,
        merge_baselines=False,
        y_lim="auto_fix",
        show_significance=True,
        line_styles={
            "female": "",
            "male": (4, 2),
        },
        figsize=(12, 8),
    ),
)

print("Trajectory config ok")

Trajectory config ok


In [7]:
workflow = PlotWorkflow(
    heatmap_config=heatmap_config,
    trajectory_config=traj_config,
)

print("Workflow ok")

Workflow ok


In [8]:
print("Heatmap metadata shape:", workflow.heatmap_meta.shape)
print("Trajectory metadata shape:", workflow.trajectory_meta.shape)

print("Sex levels:", workflow.trajectory_meta["sex"].dropna().unique())
print("Genotype levels:", workflow.trajectory_meta["mice_model"].dropna().unique())
print("Treatment levels:", workflow.trajectory_meta["description_of_treatment"].dropna().unique())

Heatmap metadata shape: (544, 72)
Trajectory metadata shape: (544, 73)
Sex levels: ['male' 'female']
Genotype levels: ['Apc' 'WT']
Treatment levels: ['sham' 'irradiated']


In [9]:
subset = SubsetSpec(
    label="WT_sham_genus_ANCOM",
    title="WT | sham",
    filters={
        "mice_model": "WT",
        "description_of_treatment": "sham",
    }
)

print("Subset ok")

Subset ok


In [10]:
workflow.plot_heatmap_and_trajectory(
    subset=subset,
    taxon_query="g_Akkermansia",
    trajectory_plot_mode="combo",
    comparison_levels=["female", "male"],
    combo_groups=[("WT", "sham")],
    heatmap_kwargs={
        "show": True,
        "save_png": False,
        "save_pdf": False,
    }
)

TypeError: PlotWorkflow.plot_heatmap_and_trajectory() got an unexpected keyword argument 'trajectory_plot_mode'